# Congestion Pricing and NYC Drivers

On 5 January 2025 New York started charging a toll to enter Manhattan below 60th Street. This notebook measures how the toll changed yellow taxi and rideshare (Uber/Lyft) trips and driver earnings.

Run it from top to bottom. The functions it calls live in `scripts/`. Setup steps are in `README.md`.

## Setup

To check that the notebook runs without processing all 36 months, uncomment `config.QUICK_RUN = True`. It then uses only January and March of 2024 and 2025.

In [ ]:
import matplotlib.pyplot as plt

from scripts import config

# config.QUICK_RUN = True

# Every figure uses the shared style
plt.style.use(config.STYLE_FILE)

months = config.months_to_process()
print(f"QUICK_RUN = {config.QUICK_RUN}: {len(months)} months, "
      f"{months[0]} to {months[-1]}")

## 1. Download the data

Everything is saved to `data/raw/`. Each download is skipped if its file is already there, so running this section again is quick. The functions are in `scripts/download.py`.

### 1.1 TLC trip data

One parquet file per month for yellow taxis and High Volume FHV (Uber/Lyft), from the [TLC trip record page](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). The full run downloads 72 files (about 20 GB), four at a time. The table shows rows × columns of every raw file, read from the parquet metadata without loading the data.

In [ ]:
from scripts import download

tlc_files = download.download_tlc(months)

# Rows x columns of every raw TLC file
tlc_shapes = download.parquet_shapes(tlc_files)
tlc_shapes["service"] = tlc_shapes["file"].str.split("_").str[0]

# Totals per service, for Table 1 in the report
tlc_shapes.groupby("service").agg(
    files=("file", "count"), rows=("rows", "sum"),
    min_columns=("columns", "min"), max_columns=("columns", "max"),
    size_gb=("size_mb", lambda mb: round(mb.sum() / 1000, 1)))

In [ ]:
# Rows x columns of each monthly file. The column count changes when the
# TLC adds a column, such as cbd_congestion_fee in 2025.
tlc_shapes.set_index("file")[["rows", "columns", "size_mb"]]

### 1.2 External datasets

Sources and notes are in `external_datasets.md`:

- **TLC taxi zone lookup and shapefile:** zone names, boroughs and shapes
- **MTA CBD taxi zones** ([data.ny.gov](https://data.ny.gov/d/yfdc-w5jh)): the 38 zones inside the tolled area
- **MTA CBD taxi/FHV speeds** ([data.ny.gov](https://data.ny.gov/d/6p29-6xqn)): monthly speeds, for one line of background
- **NOAA daily weather** for Central Park (station `USW00094728`), in metric units (mm, °C)
- **MTA subway hourly ridership** ([2023–2024](https://data.ny.gov/d/wujg-7c2s), [2025](https://data.ny.gov/d/5wq4-mkjj)): only the months in `months`

data.ny.gov returns only 1,000 rows per request by default, so the downloads set `$limit`, sort with `$order`, page with `$offset`, and check the row count against the server's own count.

The raw subway data is too big to download (about 25–45 million rows a year), so data.ny.gov groups it first: ridership is summed over payment method and fare class, leaving one row per station and hour. Grouping a whole month in one request takes minutes on their server, so it is requested one day at a time and saved one file per month. The full 36 months take about 20 minutes.

In [ ]:
external_files = download.download_external(months)

# Rows x columns of each raw external dataset (subway months added together)
download.external_shapes(external_files)

## 2. Prepare the external data

CBD zone labels, the buffer ring, subway ridership, weather and holidays. This comes before cleaning the taxi data, because the cleaning step needs the zone labels.

## 3. Clean the taxi data

Clean the trips with PySpark, recording rows × columns after every step, and save the summary table to `data/curated/`.

## 4. Explore the data and make maps

## 5. Models

Model 1: difference-in-differences regression. Model 2: LightGBM forecast of trips without the toll.